# TicketLens: Exploratory Data Analysis & Ticket Distribution
This notebook analyzes the **Tobi-Bueck/customer-support-tickets** IT helpdesk dataset, inspecting label distributions, ticket lengths, vocabulary, and PII patterns.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve() / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ticketlens.config import load_config
from ticketlens.data import load_raw_dataset, clean_and_redact_text

cfg = load_config("../configs/config.yaml")
df = load_raw_dataset(cfg)
print(f"Loaded dataset with {len(df)} rows and columns: {list(df.columns)}")
df.head(3)

### Target Category Distribution (ITIL Categories)
Analyzing the frequency of Incident, Request, Problem, and Change tickets.

In [ ]:
valid_types = ['Incident', 'Request', 'Problem', 'Change']
df_filtered = df[df['type'].isin(valid_types)].copy()

plt.figure(figsize=(8, 4))
sns.countplot(data=df_filtered, x='type', order=valid_types, palette='viridis')
plt.title('Ticket Type Frequency Distribution', fontsize=14)
plt.xlabel('Category')
plt.ylabel('Ticket Count')
plt.show()

### Text Length Distribution & PII Sanitization Inspection
Inspecting the word counts and verifying that email addresses, phone numbers, and IP addresses are properly redacted before feeding into NLP models.

In [ ]:
df_filtered['text'] = df_filtered.apply(lambda r: f"{r['subject']}\n{r['body']}", axis=1)
df_filtered['sanitized_text'] = df_filtered['text'].apply(clean_and_redact_text)
df_filtered['word_count'] = df_filtered['sanitized_text'].apply(lambda t: len(t.split()))

plt.figure(figsize=(9, 4))
sns.boxplot(data=df_filtered, x='type', y='word_count', palette='Blues')
plt.title('Ticket Word Count by Category')
plt.ylabel('Word Count')
plt.show()

print("Sample Sanitized Ticket:")
print(df_filtered['sanitized_text'].iloc[0])